In [1]:
pip install streamlit pandas matplotlib seaborn plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 43.1 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px

# Set up page styling configurations
st.set_page_config(page_title="Global Superstore Analytics", layout="wide", initial_sidebar_state="expanded")

OUTPUT_DIR = "dashboard_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, "Global_Superstore_Clean.csv")

# -------------------------------------------------------------------------
# STEP 1: RESILIENT DATA MANAGEMENT LAYER (AUTO-GENERATION FALLBACK)
# -------------------------------------------------------------------------
@st.cache_data
def load_and_prepare_data():
    if not os.path.exists(CSV_PATH):
        # Generate clean statistical global superstore profile locally
        np.random.seed(42)
        n_records = 1000

        regions = ['North America', 'Europe', 'Asia Pacific', 'LATAM', 'EMEA']
        categories = {
            'Technology': ['Phones', 'Copiers', 'Accessories', 'Machines'],
            'Furniture': ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
            'Office Supplies': ['Storage', 'Appliances', 'Art', 'Binders', 'Paper']
        }

        cat_choices = list(categories.keys())
        generated_cats = np.random.choice(cat_choices, size=n_records, p=[0.3, 0.3, 0.4])
        generated_subcats = [np.random.choice(categories[cat]) for cat in generated_cats]

        sales = np.random.lognormal(mean=5.2, sigma=1.0, size=n_records) + 5.0 # realistic sales curve
        # Profit is tied to sales dynamically with structural variance constraints
        profit_margins = np.random.uniform(-0.2, 0.4, size=n_records)
        profits = sales * profit_margins

        customer_names = [f"Customer {chr(65 + (i % 26))}{i % 15}" for i in range(n_records)]

        df_mock = pd.DataFrame({
            'Row_ID': range(1, n_records + 1),
            'Order_Date': pd.date_range(start="2024-01-01", periods=n_records, freq='h'),
            'Region': np.random.choice(regions, size=n_records),
            'Category': generated_cats,
            'Sub_Category': generated_subcats,
            'Customer_Name': np.random.choice(customer_names, size=n_records),
            'Sales': np.round(sales, 2),
            'Profit': np.round(profits, 2)
        })
        df_mock.to_csv(CSV_PATH, index=False)
        return df_mock
    else:
        df = pd.read_csv(CSV_PATH)
        df['Order_Date'] = pd.to_datetime(df['Order_Date'])
        return df

df = load_and_prepare_data()

# -------------------------------------------------------------------------
# STEP 2: SIDEBAR INTERACTIVE FILTERING CONTROLS
# -------------------------------------------------------------------------
st.sidebar.header("📊 Global Filter Controls")

# Region Filter Multi-Select
selected_regions = st.sidebar.multiselect(
    "1. Select Sales Regions",
    options=sorted(df['Region'].unique()),
    default=sorted(df['Region'].unique())
)

# Dynamic Category Multi-Select
selected_categories = st.sidebar.multiselect(
    "2. Select Product Categories",
    options=sorted(df['Category'].unique()),
    default=sorted(df['Category'].unique())
)

# Filter sub-categories dynamically based on parent selections
available_subcats = df[df['Category'].isin(selected_categories)]['Sub_Category'].unique() if selected_categories else df['Sub_Category'].unique()

selected_subcategories = st.sidebar.multiselect(
    "3. Select Product Sub-Categories",
    options=sorted(available_subcats),
    default=sorted(available_subcats)
)

# Apply Filter selections to primary dataframe array
filtered_df = df[
    (df['Region'].isin(selected_regions)) &
    (df['Category'].isin(selected_categories)) &
    (df['Sub_Category'].isin(selected_subcategories))
]

# -------------------------------------------------------------------------
# STEP 3: MAIN PANEL DASHBOARD LAYOUT & KPI WIDGETS
# -------------------------------------------------------------------------
st.title("🏆 Global Superstore Executive Dashboard")
st.markdown("Monitor and analyze corporate sales trajectories, net margin profits, and customer segmentation behaviors.")
st.markdown("---")

if filtered_df.empty:
    st.warning("⚠️ No records match the current filter selection combinations. Reset options in the sidebar.")
else:
    # Row 1: High-Value Key Performance Metrics (KPIs)
    kpi_col1, kpi_col2, kpi_col3, kpi_col4 = st.columns(4)

    total_sales = filtered_df['Sales'].sum()
    total_profit = filtered_df['Profit'].sum()
    profit_margin = (total_profit / total_sales) * 100 if total_sales > 0 else 0.0
    total_orders = filtered_df.shape[0]

    with kpi_col1:
        st.metric(label="💰 Total Revenue (Sales)", value=f"${total_sales:,.2f}")
    with kpi_col2:
        # Style indicator based on profit performance signs
        st.metric(label="📈 Net Profit Margin", value=f"${total_profit:,.2f}", delta=f"{profit_margin:.1f}% Margin")
    with kpi_col3:
        st.metric(label="📦 Volumetric Orders Count", value=f"{total_orders:,}")
    with kpi_col4:
        avg_order = filtered_df['Sales'].mean() if total_orders > 0 else 0.0
        st.metric(label="🎯 Average Ticket Value", value=f"${avg_order:,.2f}")

    st.markdown("---")

    # Row 2: Advanced Visualizations & Comparative Analytics
    chart_col1, chart_col2 = st.columns(2)

    with chart_col1:
        st.subheader("🛒 Segment Performance: Category vs Profitability")
        category_summary = filtered_df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()
        fig_bar = px.bar(
            category_summary, x='Category', y='Sales', text_auto='.2s',
            hover_data=['Profit'], color='Profit', color_continuous_scale='Viridis',
            title="Total Sales Revenue by Category (Colored by Net Profitability)"
        )
        st.plotly_chart(fig_bar, use_container_width=True)

    with chart_col2:
        st.subheader("⭐ Top 5 High-Value Customers by Sales Volume")
        top_customers = filtered_df.groupby('Customer_Name')['Sales'].sum().reset_index()
        top_customers = top_customers.sort_values(by='Sales', ascending=False).head(5)

        fig_cust = px.bar(
            top_customers, x='Sales', y='Customer_Name', orientation='h',
            text_auto='.2s', color='Sales', color_continuous_scale='Cividis',
            title="Top 5 Capital Contributing Consumer Profiles"
        )
        # Invert axis so highest billing customer appears at the top
        fig_cust.update_layout(yaxis={'categoryorder': 'total ascending'})
        st.plotly_chart(fig_cust, use_container_width=True)

    st.markdown("---")

    # Row 3: Filtered Raw Data Explorer Array
    st.subheader("🔍 Granular Transactional Level Explorer")
    st.markdown("Review and export filtered rows for external business intelligence integration toolings.")
    st.dataframe(filtered_df.sort_values(by='Order_Date', ascending=False), use_container_width=True)

    # Enable clean local downloading of modified tables directly from the browser window
    csv_buffer = filtered_df.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Filtered Slice Data as CSV",
        data=csv_buffer,
        file_name="Filtered_Superstore_Extract.csv",
        mime="text/csv"
    )

2026-05-24 12:11:17.744 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:11:17.747 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-24 12:11:17.750 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-24 12:11:17.752 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:11:17.795 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:11:17.938 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-24 12:11:17.939 Thread 'MainThread': mi